In [1]:
import pandas as pd

recs = pd.read_csv("recommendations.csv")

# elimina usuaris sin horas significativas jugadas
recs = recs[recs["hours"] > 0.1]

print("riginal:")
print("users:", recs["user_id"].nunique(), "items:", recs["app_id"].nunique())


riginal:
users: 13732369 items: 37509


In [2]:
# conteo por usuario e item (ex ante)
user_counts = recs["user_id"].value_counts()
item_counts = recs["app_id"].value_counts()

# minimo de interacciones ex ante (decidido con la distribución completa)
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

good_users = user_counts[user_counts >= MIN_USER_INTERACTIONS].index
good_items = item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index

recs_filtered = recs[
    recs["user_id"].isin(good_users) &
    recs["app_id"].isin(good_items)
]

print("después filtro global (MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS):")
print("users:", recs_filtered["user_id"].nunique(), "items:", recs_filtered["app_id"].nunique())


después filtro global (MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS):
users: 1894391 items: 34916


In [3]:
# actividad de usuario en el dataset filtrado
user_activity = recs_filtered["user_id"].value_counts().rename("n_interactions").reset_index()
user_activity.columns = ["user_id", "n_interactions"]

# creamos buckets de actividad (low / mid / high)
user_activity["activity_bucket"] = pd.qcut(
    user_activity["n_interactions"],
    q=3,  # terciles: bajo, medio, alto
    labels=["low", "mid", "high"]
)

# tamaño maximo total del sample
TARGET_USERS = 10000

# pesos deseados por bucket, mas estratificado
bucket_weights = {
    "low": 0.4,
    "mid": 0.4,
    "high": 0.2,
}

# contamos cuantos usuarios hay en cada bucket
bucket_users = {
    b: user_activity[user_activity["activity_bucket"] == b]["user_id"]
    for b in ["low", "mid", "high"]
}
bucket_counts = {b: len(u) for b, u in bucket_users.items()}

print("usuarios por bucket (después filtro global):", bucket_counts)

# 1) asignación inicial: min(deseado, disponible)
alloc = {}
for b, w in bucket_weights.items():
    desired = int(TARGET_USERS * w)
    available = bucket_counts[b]
    alloc[b] = min(desired, available)

# 2) redistribuir cupos sobrantes a buckets con capacidad extra
total_alloc = sum(alloc.values())
total_available = sum(bucket_counts.values())

if total_available <= TARGET_USERS:
    # no hay suficientes usuarios para llegar al target, tomamos todos
    final_alloc = bucket_counts.copy()
else:
    leftover = TARGET_USERS - total_alloc
    # ordenar buckets por cuánta capacidad extra tienen
    spare = {b: bucket_counts[b] - alloc[b] for b in bucket_weights.keys()}
    buckets_by_spare = sorted(spare.keys(), key=lambda b: spare[b], reverse=True)
    final_alloc = alloc.copy()
    for b in buckets_by_spare:
        if leftover <= 0:
            break
        can_take = spare[b]
        if can_take <= 0:
            continue
        take = min(can_take, leftover)
        final_alloc[b] += take
        leftover -= take

print("asignación final de usuarios por bucket:", final_alloc)
print("total usuarios muestreados:", sum(final_alloc.values()))

# 3) samplear usuarios por bucket segun la asignación final
sampled_users_list = []
for b, size in final_alloc.items():
    if size <= 0:
        continue
    sampled_users_list.append(bucket_users[b].sample(size, random_state=42))

sampled_users = pd.concat(sampled_users_list)

print("usuarios únicos muestreados:", sampled_users.nunique())


usuarios por bucket (después filtro global): {'low': 730112, 'mid': 586700, 'high': 577579}
asignación final de usuarios por bucket: {'low': 4000, 'mid': 4000, 'high': 2000}
total usuarios muestreados: 10000
usuarios únicos muestreados: 10000


In [4]:
sample_recs = recs_filtered[recs_filtered["user_id"].isin(sampled_users)]

print("después de muestrear usuarios (antes filtro de items en sample):")
print("users:", sample_recs["user_id"].nunique(), "items:", sample_recs["app_id"].nunique())


después de muestrear usuarios (antes filtro de items en sample):
users: 10000 items: 10122


In [5]:
# filtro de ítems dentro del sample
MIN_ITEM_INTERACTIONS_SAMPLE = 5

item_counts_sample = sample_recs["app_id"].value_counts()
good_items_sample = item_counts_sample[item_counts_sample >= MIN_ITEM_INTERACTIONS_SAMPLE].index

sample_recs = sample_recs[sample_recs["app_id"].isin(good_items_sample)]

print("después del filtro de ítems dentro del sample:")
print("users:", sample_recs["user_id"].nunique(), "items:", sample_recs["app_id"].nunique())


después del filtro de ítems dentro del sample:
users: 9991 items: 2880


In [6]:
sample_recs["date"] = pd.to_datetime(sample_recs["date"])
sample_recs = sample_recs.sort_values(["user_id", "date"])
sample_recs.to_csv("sampled_recommendations.csv", index=False)

n_users_final = sample_recs["user_id"].nunique()
n_items_final = sample_recs["app_id"].nunique()
print("sample FINAL:")
print(f"numero de usuarios en el sample final: {n_users_final}")
print(f"numero de ítems en el sample final: {n_items_final}")


sample FINAL:
numero de usuarios en el sample final: 9991
numero de ítems en el sample final: 2880


Generación de split de muestreo de usuarios e ítems

In [7]:
import numpy as np

df = sample_recs.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["user_id", "date"])

train_list = []
test_list = []

for user, group in df.groupby("user_id"):
    n = len(group)
    # se necesita al menos 2 interacciones para poder hacer train+test, igual esto no debería pasar por el filtro previo
    if n < 2:
        continue

    # indice de corte 80% temporal (por usuario)
    split_idx = int(np.ceil(n * 0.8))
    # por seguridad: asegurar que siempre quede al menos 1 interacción en test
    if split_idx >= n:
        split_idx = n - 1

    train_u = group.iloc[:split_idx]
    test_u  = group.iloc[split_idx:]

    train_list.append(train_u)
    test_list.append(test_u)

train = pd.concat(train_list)
test  = pd.concat(test_list)

print("train:", train.shape, "users:", train.user_id.nunique())
print("test: ", test.shape,  "users:", test.user_id.nunique())

# guardar splits
train.to_csv("train_split_8020.csv", index=False)
test.to_csv("test_split_8020.csv", index=False)


train: (73186, 8) users: 9977
test:  (15294, 8) users: 9977


In [8]:
total_interactions = len(sample_recs)
print(f"porcentaje de interacciones en train: {len(train) / total_interactions:.2%}")
print(f"porcentaje de interacciones en test:  {len(test) / total_interactions:.2%}")

porcentaje de interacciones en train: 82.70%
porcentaje de interacciones en test:  17.28%


Esto es un split aproximado 80/20 que respeta temporalidad. Sucede que el ceil redondea hacia arriba, por lo que no siempre se obtiene exactamente un 80/20. Sin embargo, el split es estratificado, ya que cada usuario aparece en ambos conjuntos (train y test).

In [9]:
train.to_csv("train_split_8020.csv", index=False)
test.to_csv("test_split_8020.csv", index=False)